In [14]:
#Experiment 17:
import math
import collections

data = [
    ["Good", "High", "Salaried", "No", "Approved"],
    ["Good", "High", "Self-employed", "No", "Approved"],
    ["Good", "Medium", "Salaried", "No", "Approved"],
    ["Good", "Low", "Salaried", "No", "Approved"],
    ["Good", "Medium", "Self-employed", "Yes", "Approved"],
    ["Good", "Low", "Self-employed", "No", "Approved"],
    ["Poor", "High", "Salaried", "No", "Approved"],
    ["Poor", "Medium", "Salaried", "No", "Approved"],
    ["Poor", "Low", "Salaried", "No", "Rejected"],
    ["Poor", "High", "Self-employed", "Yes", "Rejected"],
    ["Poor", "Medium", "Self-employed", "No", "Rejected"],
    ["Poor", "Low", "Self-employed", "Yes", "Rejected"],
    ["Good", "High", "Salaried", "Yes", "Rejected"],
    ["Good", "Medium", "Self-employed", "Yes", "Rejected"],
    ["Poor", "High", "Salaried", "Yes", "Rejected"],
    ["Poor", "Low", "Salaried", "Yes", "Rejected"]
]

features = [
    "Credit History",
    "Income",
    "Employment",
    "Existing Loan"
]

def calculate_entropy(rows):
    labels = [row[-1] for row in rows]
    counts = collections.Counter(labels)
    total = len(labels)
    entropy = 0

    for count in counts.values():
        probability = count / total
        entropy -= probability * math.log2(probability)

    return entropy

def information_gain(rows, feature_index):
    total_entropy = calculate_entropy(rows)
    groups = collections.defaultdict(list)

    for row in rows:
        groups[row[feature_index]].append(row)

    weighted_entropy = 0

    for group in groups.values():
        weighted_entropy += (
            len(group) / len(rows)
        ) * calculate_entropy(group)

    return total_entropy - weighted_entropy

def build_tree(rows, available_features):
    labels = [row[-1] for row in rows]

    if len(set(labels)) == 1:
        return labels[0]

    if not available_features:
        return collections.Counter(labels).most_common(1)[0][0]

    best_feature = max(
        available_features,
        key=lambda index: information_gain(rows, index)
    )

    tree = {features[best_feature]: {}}
    values = sorted(set(row[best_feature] for row in rows))

    for value in values:
        subset = [
            row for row in rows
            if row[best_feature] == value
        ]

        remaining_features = [
            index for index in available_features
            if index != best_feature
        ]

        tree[features[best_feature]][value] = build_tree(
            subset, remaining_features
        )

    return tree

def predict(tree, sample):
    while isinstance(tree, dict):
        attribute = next(iter(tree))
        value = sample.get(attribute)
        branches = tree[attribute]

        if value not in branches:
            return "Unknown"

        tree = branches[value]

    return tree

initial_entropy = calculate_entropy(data)

print("ID3 Decision Tree - Loan Approval")
print("----------------------------------")
print("\nInitial Entropy:", round(initial_entropy, 4))

print("\nInformation Gain:")

for index, feature in enumerate(features):
    gain = information_gain(data, index)
    print(feature, ":", round(gain, 4))

tree = build_tree(data, list(range(len(features))))

print("\nDecision Tree Root:")
print(next(iter(tree)))

samples = [
    {
        "Credit History": "Good",
        "Income": "High",
        "Employment": "Salaried",
        "Existing Loan": "No"
    },
    {
        "Credit History": "Poor",
        "Income": "Low",
        "Employment": "Self-employed",
        "Existing Loan": "Yes"
    },
    {
        "Credit History": "Poor",
        "Income": "High",
        "Employment": "Salaried",
        "Existing Loan": "No"
    }
]

print("\nLoan Application Predictions:")

for i, sample in enumerate(samples, 1):
    result = predict(tree, sample)
    print("Application", i, ":", result)


ID3 Decision Tree - Loan Approval
----------------------------------

Initial Entropy: 1.0

Information Gain:
Credit History : 0.1887
Income : 0.0182
Employment : 0.0115
Existing Loan : 0.3113

Decision Tree Root:
Existing Loan

Loan Application Predictions:
Application 1 : Approved
Application 2 : Rejected
Application 3 : Approved


In [15]:
#Experiment-18:
from collections import deque

# Locations: 0 = A, 1 = B, 2 = C
# Banana is hanging at location C

initial_state = (0, 1, False, False)
goal_location = 2

# State: (monkey_position, box_position, on_box, has_banana)

def get_actions(state):
    monkey, box, on_box, has_banana = state
    actions = []

    if has_banana:
        return actions

    # Move monkey left or right
    if not on_box:
        for direction in [-1, 1]:
            new_position = monkey + direction

            if 0 <= new_position <= 2 and new_position != box:
                actions.append((
                    (new_position, box, False, False),
                    "Move to location " + str(new_position)
                ))

    # Push the box one step away from the monkey
    if not on_box and abs(monkey - box) == 1:
        direction = 1 if box > monkey else -1
        new_box_position = box + direction

        if 0 <= new_box_position <= 2:
            actions.append((
                (box, new_box_position, False, False),
                "Push the box to location " + str(new_box_position)
            ))

    # Climb onto the box
    if not on_box and abs(monkey - box) == 1 and box == goal_location:
        actions.append((
            (box, box, True, False),
            "Climb onto the box"
        ))

    # Grasp the banana
    if on_box and monkey == goal_location and box == goal_location:
        actions.append((
            (monkey, box, True, True),
            "Grasp the banana"
        ))

    return actions


def solve_monkey_problem():
    queue = deque([(initial_state, [])])
    visited = {initial_state}

    while queue:
        current_state, path = queue.popleft()

        if current_state[3]:
            return path

        for next_state, action in get_actions(current_state):
            if next_state not in visited:
                visited.add(next_state)
                queue.append((next_state, path + [action]))

    return None


# Execute the planning algorithm
solution = solve_monkey_problem()

print("Monkey and Banana Problem")
print("-------------------------")
print("Initial State: Monkey at A, Box at B")
print("Banana Location: C")

print("\nSequence of Actions:")

if solution:
    for step, action in enumerate(solution, 1):
        print("Step", step, ":", action)

    print("\nGoal Achieved: Monkey obtained the banana!")
    print("Total Actions:", len(solution))

else:
    print("No solution found.")


Monkey and Banana Problem
-------------------------
Initial State: Monkey at A, Box at B
Banana Location: C

Sequence of Actions:
Step 1 : Push the box to location 2
Step 2 : Climb onto the box
Step 3 : Grasp the banana

Goal Achieved: Monkey obtained the banana!
Total Actions: 3


In [16]:
#Experiment 19:
move_count = 0
def tower_of_hanoi(n, source, auxiliary, destination):
    global move_count
    if n == 1:
        move_count += 1
        print(
            "Move disk 1 from",
            source,
            "to",
            destination
        )
        return
    tower_of_hanoi(
        n - 1,
        source,
        destination,
        auxiliary
    )
    move_count += 1
    print(
        "Move disk", n,
        "from", source,
        "to", destination
    )
    tower_of_hanoi(
        n - 1,
        auxiliary,
        source,
        destination
    )
n = 3
print("Towers of Hanoi")
print("----------------")
print("Number of disks:", n)
print("\nSequence of moves:")
tower_of_hanoi(n, 'A', 'B', 'C')
print("\nTotal Moves:", move_count)
print("Minimum Moves Required:", 2**n - 1)

Towers of Hanoi
----------------
Number of disks: 3

Sequence of moves:
Move disk 1 from A to C
Move disk 2 from A to B
Move disk 1 from C to B
Move disk 3 from A to C
Move disk 1 from B to A
Move disk 2 from B to C
Move disk 1 from A to C

Total Moves: 7
Minimum Moves Required: 7


In [17]:
#Experiment 20:
rooms = {
    'A': 'Dirty',
    'B': 'Clean',
    'C': 'Dirty',
    'D': 'Dirty'
}

locations = ['A', 'B', 'C', 'D']
current_position = 0

def simple_reflex_agent(location, status):
    if status == 'Dirty':
        return 'Suck'
    return 'Move'

print("Simple Reflex Agent - Vacuum World")
print("-----------------------------------")
print("Initial Room Conditions:", rooms)

step = 1

while True:
    if all(status == 'Clean' for status in rooms.values()):
        print("\nAll rooms are clean!")
        print("Cleaning task completed.")
        break

    current_room = locations[current_position]
    current_status = rooms[current_room]

    action = simple_reflex_agent(
        current_room,
        current_status
    )

    print(
        "\nStep", step,
        "- Location:", current_room,
        "- Status:", current_status
    )

    # Execute action
    if action == 'Suck':
        rooms[current_room] = 'Clean'
        print("Action: Cleaning room", current_room)

    else:
        print("Action: Moving to next room")
        current_position = (
            current_position + 1
        ) % len(locations)

    step += 1

print("\nFinal Room Conditions:", rooms)
print("Total Actions Performed:", step - 1)


Simple Reflex Agent - Vacuum World
-----------------------------------
Initial Room Conditions: {'A': 'Dirty', 'B': 'Clean', 'C': 'Dirty', 'D': 'Dirty'}

Step 1 - Location: A - Status: Dirty
Action: Cleaning room A

Step 2 - Location: A - Status: Clean
Action: Moving to next room

Step 3 - Location: B - Status: Clean
Action: Moving to next room

Step 4 - Location: C - Status: Dirty
Action: Cleaning room C

Step 5 - Location: C - Status: Clean
Action: Moving to next room

Step 6 - Location: D - Status: Dirty
Action: Cleaning room D

All rooms are clean!
Cleaning task completed.

Final Room Conditions: {'A': 'Clean', 'B': 'Clean', 'C': 'Clean', 'D': 'Clean'}
Total Actions Performed: 6
